In [3]:
# pip install ruptures

In [7]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# Full GGS-like Change Point Detection
# ------------------------------------------------------------
# Input  : ./tvpvar_input_scaled.csv
# Output : ./result/
#
# Purpose:
# - 전체 feature의 공동 분포 구조 변화 탐지
# - Full market-wide regime change 탐지
# - penalty / model / min_size grid search
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

DATE_COL = "Date"

FEATURES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
    "d_PJM_LMP",
]

# Full GGS는 market-wide regime 탐지가 목적이므로 너무 많은 cp는 피함
TARGET_MIN_CP = 3
TARGET_MAX_CP = 7

# 후보 h.p.
MODEL_LIST = ["rbf", "normal", "l2"]
PENALTY_LIST = [0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30, 50]
MIN_SIZE_LIST = [10, 20, 30, 40, 60]

# output files
OUT_GRID = RESULT_DIR / "full_ggs_grid_search.csv"
OUT_CP = RESULT_DIR / "full_ggs_change_points.csv"
OUT_SEGMENTS = RESULT_DIR / "full_ggs_segments.csv"
OUT_SEGMENT_STATS = RESULT_DIR / "full_ggs_segment_stats.csv"
OUT_PLOT_ALL = RESULT_DIR / "full_ggs_change_points_all_features.png"

# -----------------------------
# Helpers
# -----------------------------
def validate_input(df):
    if DATE_COL not in df.columns:
        raise ValueError(f"'{DATE_COL}' column not found.")

    missing_cols = [c for c in FEATURES if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing feature columns: {missing_cols}")


def load_data():
    df = pd.read_csv(INPUT_FILE)
    validate_input(df)

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    data = df[[DATE_COL] + FEATURES].copy()
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(how="any")
    data = data.sort_values(DATE_COL).reset_index(drop=True)

    if data.empty:
        raise ValueError("No usable data after dropna.")

    return data


def run_segmentation(X, model_name, penalty, min_size):
    try:
        model = rpt.Pelt(model=model_name, min_size=min_size).fit(X)
        cps = model.predict(pen=penalty)

        # ruptures는 마지막 len(X)를 포함하므로 제외
        change_idx = [int(cp) for cp in cps if cp < len(X)]

        return change_idx, ""

    except Exception as e:
        return [], repr(e)


def score_candidate(n_cp):
    """
    목표 cp 개수에 가까울수록 낮은 score.

    우선순위:
    1. TARGET_MIN_CP ~ TARGET_MAX_CP 범위 안이면 가장 좋음
    2. 0개는 피함
    3. 너무 많은 cp는 감점
    """
    if TARGET_MIN_CP <= n_cp <= TARGET_MAX_CP:
        return 0

    if n_cp == 0:
        return 100

    if n_cp < TARGET_MIN_CP:
        return 10 + (TARGET_MIN_CP - n_cp)

    return 20 + (n_cp - TARGET_MAX_CP)


def choose_best_candidate(grid_df):
    valid = grid_df[grid_df["error_msg"] == ""].copy()

    if valid.empty:
        raise RuntimeError("All GGS candidates failed.")

    valid["score"] = valid["n_cp"].apply(score_candidate)

    target_mid = (TARGET_MIN_CP + TARGET_MAX_CP) / 2
    valid["distance_to_target_mid"] = (valid["n_cp"] - target_mid).abs()

    valid = valid.sort_values(
        ["score", "distance_to_target_mid", "min_size", "penalty"],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    return valid.iloc[0]


def make_change_point_table(change_idx, dates):
    return pd.DataFrame({
        "cp_id": np.arange(1, len(change_idx) + 1),
        "change_idx": change_idx,
        "change_date": dates.iloc[change_idx].values,
    })


def make_segment_table(change_idx, data):
    dates = data[DATE_COL]
    bounds = [0] + change_idx + [len(data)]

    rows = []

    for seg_id in range(len(bounds) - 1):
        start_idx = bounds[seg_id]
        end_idx_exclusive = bounds[seg_id + 1]
        end_idx = end_idx_exclusive - 1

        rows.append({
            "segment_id": seg_id,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "start_date": dates.iloc[start_idx],
            "end_date": dates.iloc[end_idx],
            "length": end_idx_exclusive - start_idx,
        })

    return pd.DataFrame(rows)


def make_segment_stats(data, segments_df):
    rows = []

    for _, seg in segments_df.iterrows():
        seg_id = int(seg["segment_id"])
        start_idx = int(seg["start_idx"])
        end_idx = int(seg["end_idx"])

        seg_data = data.iloc[start_idx:end_idx + 1]

        for feature in FEATURES:
            values = seg_data[feature].values

            rows.append({
                "segment_id": seg_id,
                "feature": feature,
                "mean": float(np.mean(values)),
                "std": float(np.std(values, ddof=0)),
                "var": float(np.var(values, ddof=0)),
                "min": float(np.min(values)),
                "max": float(np.max(values)),
                "median": float(np.median(values)),
            })

    return pd.DataFrame(rows)


def plot_all_features(data, change_idx, out_path):
    dates = data[DATE_COL]

    plt.figure(figsize=(15, 7))

    for feature in FEATURES:
        plt.plot(
            dates,
            data[feature],
            linewidth=0.8,
            alpha=0.55,
            label=feature,
        )

    for cp in change_idx:
        plt.axvline(
            dates.iloc[cp],
            linestyle="--",
            linewidth=1.2,
            alpha=0.8,
        )

    plt.title("Full GGS-like Change Points - All Features")
    plt.xlabel("Date")
    plt.ylabel("Standardized value")
    plt.legend(loc="upper left", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


# -----------------------------
# Main
# -----------------------------
def main():
    print("==========================================")
    print("Full GGS-like Change Point Detection")
    print("==========================================")

    data = load_data()
    X = data[FEATURES].values
    dates = data[DATE_COL]

    print("Input file :", INPUT_FILE)
    print("Rows       :", len(data))
    print("Features   :", len(FEATURES))
    print("Date range :", dates.min(), "~", dates.max())
    print()

    # -----------------------------
    # Grid search
    # -----------------------------
    grid_rows = []

    print("[1] Running grid search...")

    for model_name in MODEL_LIST:
        for min_size in MIN_SIZE_LIST:
            for penalty in PENALTY_LIST:
                change_idx, error_msg = run_segmentation(
                    X=X,
                    model_name=model_name,
                    penalty=penalty,
                    min_size=min_size,
                )

                grid_rows.append({
                    "model": model_name,
                    "min_size": min_size,
                    "penalty": penalty,
                    "n_cp": len(change_idx),
                    "change_idx": "|".join(map(str, change_idx)),
                    "change_dates": "|".join(
                        dates.iloc[change_idx].astype(str).tolist()
                    ) if len(change_idx) > 0 else "",
                    "error_msg": error_msg,
                })

    grid_df = pd.DataFrame(grid_rows)
    grid_df.to_csv(OUT_GRID, index=False, encoding="utf-8-sig")

    print("Saved grid search:", OUT_GRID)

    # -----------------------------
    # Choose best h.p.
    # -----------------------------
    best = choose_best_candidate(grid_df)

    best_model = best["model"]
    best_min_size = int(best["min_size"])
    best_penalty = float(best["penalty"])

    change_idx, error_msg = run_segmentation(
        X=X,
        model_name=best_model,
        penalty=best_penalty,
        min_size=best_min_size,
    )

    if error_msg:
        raise RuntimeError(f"Best candidate failed unexpectedly: {error_msg}")

    print()
    print("[2] Selected setting")
    print("model    :", best_model)
    print("min_size :", best_min_size)
    print("penalty  :", best_penalty)
    print("n_cp     :", len(change_idx))
    print("cp idx   :", change_idx)
    print()

    if len(change_idx) == 0:
        print("[WARN] No change point detected even after grid search.")
        print("       Full GGS did not find robust market-wide regime shifts.")

    # -----------------------------
    # Make outputs
    # -----------------------------
    cp_df = make_change_point_table(change_idx, dates)
    segments_df = make_segment_table(change_idx, data)
    stats_df = make_segment_stats(data, segments_df)

    cp_df.to_csv(OUT_CP, index=False, encoding="utf-8-sig")
    segments_df.to_csv(OUT_SEGMENTS, index=False, encoding="utf-8-sig")
    stats_df.to_csv(OUT_SEGMENT_STATS, index=False, encoding="utf-8-sig")

    plot_all_features(data, change_idx, OUT_PLOT_ALL)

    # -----------------------------
    # Print summary
    # -----------------------------
    print("[3] Change points")
    if cp_df.empty:
        print("No change points detected.")
    else:
        print(cp_df.to_string(index=False))

    print()
    print("[4] Segments")
    print(segments_df.to_string(index=False))

    print()
    print("[5] Segment stats preview")
    print(stats_df.head(20).to_string(index=False))

    print()
    print("Saved files:")
    print(" -", OUT_GRID)
    print(" -", OUT_CP)
    print(" -", OUT_SEGMENTS)
    print(" -", OUT_SEGMENT_STATS)
    print(" -", OUT_PLOT_ALL)

    print()
    print("Finished.")


if __name__ == "__main__":
    main()

Full GGS-like Change Point Detection
Input file : tvpvar_input_scaled.csv
Rows       : 606
Features   : 10
Date range : 2022-12-01 00:00:00 ~ 2026-01-06 00:00:00

[1] Running grid search...
Saved grid search: result\full_ggs_grid_search.csv

[2] Selected setting
model    : l2
min_size : 60
penalty  : 10.0
n_cp     : 5
cp idx   : [85, 175, 240, 410, 470]

[3] Change points
 cp_id  change_idx change_date
     1          85  2023-05-09
     2         175  2023-10-23
     3         240  2024-02-21
     4         410  2025-01-06
     5         470  2025-04-28

[4] Segments
 segment_id  start_idx  end_idx start_date   end_date  length
          0          0       84 2022-12-01 2023-05-08      85
          1         85      174 2023-05-09 2023-10-19      90
          2        175      239 2023-10-23 2024-02-20      65
          3        240      409 2024-02-21 2025-01-02     170
          4        410      469 2025-01-06 2025-04-24      60
          5        470      605 2025-04-28 2026-01-06